
# Building Multi-Agent Systems with Strands Agent Graph
Multi-agent systems leverage multiple specialized AI agents working together to solve complex problems through coordinated collaboration. Each agent has specific capabilities and roles, connected through explicit communication pathways.

In this lab, you'll learn to build multi-agent systems using the Strands Agent SDK. We'll progress from basic concepts to advanced implementations, exploring different topologies and real-world applications.

**Learning Objectives:**
By the end of this notebook, you'll be able to:
- Understand the three core components of agent graphs (nodes, edges, conditions)
- Send targeted messages between specific agents
- Monitor and control multi-agent networks
- Design specialized agent systems for real-world scenarios

## Prerequisites

- Python 3.10+
- AWS account with Anthropic Claude Sonnet 4 enabled on Amazon Bedrock
- IAM role with permissions to use Amazon Bedrock
- Basic understanding of AI agents and prompt engineering

## Setup and Installation

Before we start, let's install the requirement packages for `strands-agents` and `strands-agents-tools`

In [2]:
!pip install -r requirements.txt

  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-macosx_11_0_arm64.whl.metadata (2.4 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached jmespath-1.1.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.8.0-cp314-cp314-macosx_11_0_arm64.whl.metadata (20 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.3/728.3 kB 32.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 69.7 MB/s  0:00:00 eta 0:0

### Importing required packages

Next we can import the required packages

In [3]:
from strands import Agent

## Understanding Agent Graph Components
An agent graph is a structured network of interconnected AI agents designed to solve complex problems through coordinated collaboration. Each agent represents a specialized node with specific capabilities, and the connections between agents define explicit communication pathways.

Before we start building, let's understand the three primary components of an agent graph:

### 1. Nodes (Agents)
Each node represents an AI agent with:
- **Identity**: Unique identifier within the graph
- **Role**: Specialized function or purpose
- **System Prompt**: Instructions defining the agent's behavior
- **Tools**: Capabilities available to the agent

### 2. Edges (Connections)
Edges define communication pathways with:
- **Direction**: One-way or bidirectional information flow
- **Condition**: Optional function that determines if the edge should be traversed
- **Dependencies**: Define execution order and data flow between nodes

### 3. GraphBuilder
The GraphBuilder provides a simple interface for constructing graphs:
- **add_node()**: Add an agent or multi-agent system as a node
- **add_edge()**: Create a dependency between nodes
- **set_entry_point()**: Define starting nodes for execution
- **build()**: Validate and create the Graph instance



### Basic processing

Let's start with a simple example of one task processed by two different agents providing an output that will depend on their defined role. Take a look at the execution order of the nodes and also the fact that with Strands SDK you can explicitly get a response only from one single node if needed. Architecture looks as following:

<div style="text-align:left">
    <img src="images/basic.png" width="55%" />
</div>

In [4]:
#Initialize an agent with agent_graph capability
from strands.multiagent import GraphBuilder

# Create specialized agents
coordinator = Agent(name="coordinator", system_prompt="You are a research team leader coordinating specialists. Provide a short analysis, no need for follow ups")
analyst = Agent(name="data_analyst", system_prompt="You are a data analyst specializing in statistical analysis. Provide a short analysis, no need for follow ups")
domain_expert = Agent(name="domain_expert", system_prompt="You are a domain expert with deep subject knowledge. Provide a short analysis, no need for follow ups")

# Build the graph
builder = GraphBuilder()

# Add nodes
builder.add_node(coordinator, "team_lead")
builder.add_node(analyst, "analyst")
builder.add_node(domain_expert, "expert")

# Add edges (dependencies)
builder.add_edge("team_lead", "analyst")
builder.add_edge("team_lead", "expert")

# Set entry points (optional - will be auto-detected if not specified)
builder.set_entry_point("team_lead")

# Build the graph
graph = builder.build()

#Execute task on newly built graph
result = graph("Analyze the impact of remote work on employee productivity.Provide a short analysis, no need for follow ups")
print("\n")
print("============================================================")
print("============================================================")

print(f"Response: {result}")

print("=============Node execution order:==========================")
print("============================================================")

# See which nodes were executed and in what order
for node in result.execution_order:
    print(f"Executed: {node.node_id}")

print("=============Graph metrics:=================================")
print("============================================================")


# Get performance metrics
print(f"Total nodes: {result.total_nodes}")
print(f"Completed nodes: {result.completed_nodes}")
print(f"Failed nodes: {result.failed_nodes}")
print(f"Execution time: {result.execution_time}ms")
print(f"Token usage: {result.accumulated_usage}")


# Get results from specific nodes
print("\n")
print("=============Expert node results only:======================")
print("============================================================")
print(result.results["expert"].result)

Graph without execution limits may run indefinitely if cycles exist


# Impact of Remote Work on Employee Productivity

## Key Findings

### ✅ Positive Impacts
- **Reduced commute time** converts to focused work hours
- **Flexible scheduling** allows work during peak personal productivity windows
- **Fewer office interruptions** improves deep work and concentration
- Studies (Stanford, Microsoft) show **13-40% productivity gains** in certain roles
- Higher autonomy correlates with **increased job satisfaction and retention**

### ❌ Negative Impacts
- **Collaboration deficits** slow creative and complex problem-solving
- **Communication delays** affect team-dependent workflows
- **Home distractions** (family, environment) reduce output quality
- **Technology fatigue** from excessive virtual meetings
- Blurred work-life boundaries lead to **burnout over time**

---

## Critical Variables
| Factor | High Performance | Low Performance |
|--------|-----------------|-----------------|
| Role Type | Independent/technical | Highly collaborative |
| Home Environm

### Parallel processing

Now let's create a topology when we will have 2 agents processing the request looking at 2 different aspects of the problem and have them input into a final agent responsible for summarization and risk calculation based on provided input 
<div style="text-align:left">
    <img src="images/parallel.png" width="55%" />
</div>

In [5]:
#Initialize an agent with agent_graph capability
from strands.multiagent import GraphBuilder

# Create specialized agents

financial_advisor = Agent(name="financial_advisor", system_prompt="You are a financial advisor focused on cost-benefit analysis, budget implications, and ROI calculations. Engage with other experts to build comprehensive financial perspectives.")
technical_architect = Agent(name="technical_architect", system_prompt="You are a technical architect who evaluates feasibility, implementation challenges, and technical risks. Collaborate with other experts to ensure technical viability.")
market_researcher = Agent(name="market_researcher", system_prompt="You are a market researcher who analyzes market conditions, user needs, and competitive landscape. Work with other experts to validate market opportunities.")
risk_analyst = Agent(name="risk_analyst", system_prompt="You are a risk analyst who identifies potential risks, mitigation strategies, and compliance issues. Collaborate with other experts to ensure comprehensive risk assessment.")


# Build the graph
builder = GraphBuilder()

# Add nodes
builder.add_node(financial_advisor, "finance_expert")
builder.add_node(technical_architect, "tech_expert")
builder.add_node(market_researcher, "market_expert")
builder.add_node(risk_analyst, "risk_analyst")

# Add edges (dependencies)
builder.add_edge("finance_expert", "tech_expert")
builder.add_edge("finance_expert", "market_expert")
builder.add_edge("tech_expert", "risk_analyst")
builder.add_edge("market_expert", "risk_analyst")


# Set entry points (optional - will be auto-detected if not specified)
builder.set_entry_point("finance_expert")

# Build the graph
graph = builder.build()

print("============================================================")
print("============================================================")

#Execute task on newly built graph
result = graph("Our company is considering launching a new AI-powered customer service platform. Initial investment is \$2M with projected 3-year ROI of 150%. What's your financial assessment?")
print("\n")
print("============================================================")
print("============================================================")

print(f"Response: {result}")

print("=============Node execution order:==========================")
print("============================================================")

# See which nodes were executed and in what order
for node in result.execution_order:
    print(f"Executed: {node.node_id}")

print("=============Graph metrics:=================================")
print("============================================================")


# Get performance metrics
print(f"Total nodes: {result.total_nodes}")
print(f"Completed nodes: {result.completed_nodes}")
print(f"Failed nodes: {result.failed_nodes}")
print(f"Execution time: {result.execution_time}ms")
print(f"Token usage: {result.accumulated_usage}")


# Get results from specific nodes

print("Financial Advisor:")
print("============================================================")
print("============================================================")
print(result.results["finance_expert"].result)
print("\n")

print("Technical Expert:")
print("============================================================")
print("============================================================")
print(result.results["tech_expert"].result)
print("\n")

print("Market Researcher:")
print("============================================================")
print("============================================================")
print(result.results["market_expert"].result)
print("\n")

<>:38: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
<>:38: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
/var/folders/hq/0gqg_g051vngvm87whnx3dcr0000gn/T/ipykernel_54292/1461822200.py:38: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
  result = graph("Our company is considering launching a new AI-powered customer service platform. Initial investment is \$2M with projected 3-year ROI of 150%. What's your financial assessment?")
Graph without execution limits may run indefinitely if cycles exist


# Financial Assessment: AI Customer Service Platform

## Initial ROI Validation

Let me stress-test that 150% ROI claim before you commit $2M.

### The Basic Math
| Metric | Value |
|--------|-------|
| Initial Investment | $2,000,000 |
| Claimed ROI (150%) | $3,000,000 return |
| Required Total Value Generated | $5,000,000 |
| Annual Value Needed | ~$1,667,000/year |

**That's a high bar that requires serious scrutiny.**

---

## Critical Questions About That ROI Figure

**What's actually included in the $2M?**
- Implementation costs only? Or full TCO?
- Often missing: ongoing licensing, maintenance (typically 15-20% annually), staff retraining, integration costs
- **Real cost is likely $2.8-3.2M over 3 years**

**How is the $3M return calculated?**
- Labor cost reduction (most common claim - verify headcount assumptions)
- Revenue from improved customer satisfaction
- Reduced churn value
- These are frequently **optimistic projections**

---

## Honest Risk Factors

### Downside Scen

### Branching with conditions

Let's create an agent graph that would classify the request and depending on conditions we define in the code - will route the request either to technical or business agent.

Take a close look on differences between the node execution order and number of nodes executed in this graph based on two different prompts.

<div style="text-align:left">
    <img src="images/conditional.png" width="55%" />
</div>

In [6]:
#Initialize an agent with agent_graph capability
from strands.multiagent import GraphBuilder

# Create specialized agents

classifier = Agent(name="classifier", system_prompt="You are an agent responsible for classification of the report request, return only Technical or Business classification.")
technical_report = Agent(name="technical_expert", system_prompt="You are a technical expert that focuses on providing short summary from technical perspective")
business_report = Agent(name="business_expert", system_prompt="You are a business expert that focuses on providing short summary from business perspective")

# Build the graph
builder = GraphBuilder()

# Add nodes
builder.add_node(classifier, "classifier")
builder.add_node(technical_report, "technical_report")
builder.add_node(business_report, "business_report")

def is_technical(state):
    classifier_result = state.results.get("classifier")
    if not classifier_result:
        return False
    result_text = str(classifier_result.result)
    return "technical" in result_text.lower()

def is_business(state):
    classifier_result = state.results.get("classifier")
    if not classifier_result:
        return False
    result_text = str(classifier_result.result)
    return "business" in result_text.lower()

# Add edges (dependencies)
builder.add_edge("classifier", "technical_report", condition=is_technical)
builder.add_edge("classifier", "business_report", condition=is_business)

# Set entry points (optional - will be auto-detected if not specified)
builder.set_entry_point("classifier")

# Build the graph
graph = builder.build()

print("============================================================")
print("============================================================")

#Execute task on newly built graph
result = graph("Provide report on technical aspect of working from home, outline things to consider and key risk factors")
print("\n")
print("============================================================")
print("============================================================")

print(f"Response: {result}")

print("=============Node execution order:==========================")
print("============================================================")

# See which nodes were executed and in what order
for node in result.execution_order:
    print(f"Executed: {node.node_id}")

print("=============Graph metrics:=================================")
print("============================================================")


# Get performance metrics
print(f"Total nodes: {result.total_nodes}")
print(f"Completed nodes: {result.completed_nodes}")
print(f"Failed nodes: {result.failed_nodes}")
print(f"Execution time: {result.execution_time}ms")
print(f"Token usage: {result.accumulated_usage}")

# Get results from specific nodes

print("Classifier:")
print("============================================================")
print("============================================================")
print(result.results["classifier"].result)
print("\n")

#Execute task on newly built graph
result = graph("Provide report on business impact of working from home, outline things to consider and key risk factors")
print("\n")
print("============================================================")
print("============================================================")

print(f"Response: {result}")

print("=============Node execution order:==========================")
print("============================================================")

# See which nodes were executed and in what order
for node in result.execution_order:
    print(f"Executed: {node.node_id}")

print("=============Graph metrics:=================================")
print("============================================================")


# Get performance metrics
print(f"Total nodes: {result.total_nodes}")
print(f"Completed nodes: {result.completed_nodes}")
print(f"Failed nodes: {result.failed_nodes}")
print(f"Execution time: {result.execution_time}ms")
print(f"Token usage: {result.accumulated_usage}")

# Get results from specific nodes

print("Classifier:")
print("============================================================")
print("============================================================")
print(result.results["classifier"].result)
print("\n")

Graph without execution limits may run indefinitely if cycles exist


**Technical**# Technical Report: Working From Home – Key Considerations & Risk Factors

---

## 1. Network & Connectivity Infrastructure

**Considerations:**
- Bandwidth requirements for video conferencing, VPNs, and cloud applications
- Router capability and Wi-Fi standards (Wi-Fi 5 vs Wi-Fi 6)
- ISP reliability and SLA guarantees

**Key Risks:**
- Insufficient upload/download speeds causing productivity loss
- Shared residential bandwidth contention
- Single point of failure with no redundant connection

---

## 2. Security & Cybersecurity

**Considerations:**
- VPN implementation and split-tunneling configurations
- Endpoint protection (EDR/antivirus) on all devices
- Multi-Factor Authentication (MFA) enforcement
- Network segmentation (separating work devices from IoT/personal devices)

**Key Risks:**
- Unsecured home networks exposing corporate data
- Phishing and social engineering attacks (elevated in remote environments)
- Shadow IT — unauthorized applications and tools
- Unpat

## Key Takeaways and Best Practices


### Best Practices:

**Design for acyclicity:** Ensure your graph has no cycles</p>
**Use meaningful node IDs:** Choose descriptive names for nodes</p>
**Validate graph structure:** The builder will check for cycles and validate entry points</p>
**Handle node failures:** Consider how failures in one node affect the overall workflow</p>
**Use conditional edges:** For dynamic workflows based on intermediate results</p>
**Consider parallelism:** Independent branches can execute concurrently</p>
**Nest multi-agent patterns:** Use Swarms within Graphs for complex workflows</p>
**Leverage multi-modal inputs:** Use ContentBlocks for rich inputs including images</p>

## Conclusion

You've now mastered the fundamentals of building multi-agent systems with Strands Agent Graph! You can create sophisticated networks of specialized AI agents that collaborate to solve complex problems.

The key to successful multi-agent systems is:
- Matching topology to use case
- Defining clear agent roles and responsibilities  
- Establishing proper communication patterns
- Managing resources and cleanup effectively

From here, you can build increasingly sophisticated systems for real-world applications in research, content creation, decision-making, customer service, and beyond.